In [0]:
%sql
-- 1. Nos paramos en la base de datos correcta
USE control_produccion_pozos;

-- 2. EL MERGE EN SQL BLINDADO CON TRY_CAST (A prueba de fallas reales)
MERGE INTO pozos_plata AS destino
USING (
    -- Subquery para limpiar y formatear los datos crudos de Bronce
    SELECT 
        id_pozo,
        
        -- Convertimos el string de fecha de forma segura
        TO_TIMESTAMP(timestamp_sensor, 'yyyy-MM-dd HH:mm:ss') AS fecha_hora,
        
        -- SOLUCIÓN: Usamos TRY_CAST para tolerar texto malformado y devolver NULL sin explotar
        TRY_CAST(presion_cabezal_raw AS FLOAT) AS presion_psi,
        TRY_CAST(caudal_bpd_raw AS FLOAT) AS caudal_bpd,
        
        -- Evaluamos la calidad del dato para tu dashboard (Criterio Senior)
        CASE 
            WHEN TRY_CAST(presion_cabezal_raw AS FLOAT) IS NULL 
                 OR TRY_CAST(caudal_bpd_raw AS FLOAT) IS NULL 
                 OR TRY_CAST(presion_cabezal_raw AS FLOAT) < 0.0 
                 OR timestamp_sensor IS NULL 
                 OR timestamp_sensor = '' THEN 'ERROR_LECTURA'
            ELSE 'OK'
        END AS estado_sensor,
        
        CURRENT_TIMESTAMP() AS fecha_ingesta_plata
    FROM pozos_bronce
) AS nuevos_datos
ON destino.id_pozo = nuevos_datos.id_pozo AND destino.fecha_hora = nuevos_datos.fecha_hora

-- Si el registro ya existe (mismo pozo, misma hora), pisamos los valores por seguridad (No duplica)
WHEN MATCHED THEN
  UPDATE SET 
    destino.presion_psi = nuevos_datos.presion_psi,
    destino.caudal_bpd = nuevos_datos.caudal_bpd,
    destino.estado_sensor = nuevos_datos.estado_sensor,
    destino.fecha_ingesta_plata = nuevos_datos.fecha_ingesta_plata

-- Si la combinación pozo + hora es nueva y la fecha es válida, la insertamos
WHEN NOT MATCHED AND nuevos_datos.fecha_hora IS NOT NULL THEN
  INSERT (id_pozo, fecha_hora, presion_psi, caudal_bpd, estado_sensor, fecha_ingesta_plata)
  VALUES (nuevos_datos.id_pozo, nuevos_datos.fecha_hora, nuevos_datos.presion_psi, nuevos_datos.caudal_bpd, nuevos_datos.estado_sensor, nuevos_datos.fecha_ingesta_plata);


In [0]:
%sql
SELECT * FROM pozos_plata ORDER BY fecha_hora;